# 🎓 Student Guide - Live Session Prep

## Before the Session Starts:

### 1️⃣ Get Your API Keys

**OpenAI API Key** (Required):
- Go to: https://platform.openai.com/api-keys
- Sign in or create an account
- Click "Create new secret key"
- Copy the key (starts with `sk-...`)

**LangSmith API Key** (Optional - for tracing):
- Go to: https://smith.langchain.com
- Sign in or create account
- Navigate to Settings → API Keys
- Click "Create API Key"
- Copy the key (starts with `lsv2_...`)

### 2️⃣ Create .env File

In the same folder as this notebook, create a file named `.env` with:

```
OPENAI_API_KEY=sk-your-key-here
LANGSMITH_API_KEY=lsv2_your-key-here
LANGSMITH_TRACING_V2=true
LANGSMITH_PROJECT=YouTube-Transcript-Summarizer
```

### 3️⃣ During the Live Session:

- Follow along as we code each section
- Run the install cell when prompted
- Test with a YouTube URL that has English captions
- Ask questions anytime!

# YouTube Transcript Summarizer with LangChain

## 📚 Project Goal
Summarize video content by extracting transcripts from YouTube videos and using LLMs (via LangChain) to generate structured, concise summaries.

## 🎯 Key Outcomes
- **Extract and preprocess transcripts** from YouTube videos using `yt_dlp`
- **Use LLMs to generate structured summaries** with LangChain
- **Build reusable components** for transcript extraction and summarization
- **Handle errors gracefully** and process multiple videos efficiently

## 1️⃣ Setup & Dependencies

In [ ]:
# ============================================================================
# CELL 1: INSTALL REQUIRED PACKAGES
# ============================================================================
# Why this step?
# Python projects need external libraries to work. This cell installs them.
# Think of it like: "Download and install all tools we'll use"
#
# What are we installing?
# - langchain & langchain-openai: AI framework for working with LLMs
# - langchain-community: Additional LangChain tools
# - yt-dlp: Download YouTube video metadata (captions, title, etc.)
# - python-dotenv: Load API keys from .env file (keeps secrets safe)
# - langsmith: (Optional) Monitor & trace our AI pipeline
# ============================================================================

import sys
import subprocess

# List of packages to install
packages = [
    "langchain",
    "langchain-community",
    "langchain-openai",
    "langsmith",
    "yt-dlp",
    "python-dotenv"
]

# Install each package quietly (no verbose output)
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

print(f"✓ Installed {len(packages)} packages successfully!")


In [ ]:
# ============================================================================
# CELL 2: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# In Python, we import libraries before using them.
# Think of imports like: "Show me where the tools are so I can use them"
#
# We'll use these libraries:
# 1. File & Data: os (environment), json (parse data), re (pattern matching)
# 2. Web: requests (download files)
# 3. Time: datetime (timestamps for logging)
# 4. LangChain: AI framework for building pipelines
# 5. YouTube: yt_dlp (extract captions)
# 6. Environment: python-dotenv (load secrets from .env file)
# ============================================================================

# STEP 1: Import standard Python libraries
import os
import json
import re
import pandas as pd
import requests
import subprocess
import sys
from dotenv import load_dotenv  # Load API keys from .env file
from datetime import datetime   # Get current time for logging

# STEP 2: Load environment variables from .env file
# This reads your .env file and puts variables into os.environ
# So os.getenv("OPENAI_API_KEY") will return your actual API key
load_dotenv()

# Get your OpenAI API key from the environment
api_key = os.getenv("OPENAI_API_KEY")

# Validate that the API key exists before proceeding
# If it's missing, show a helpful error message now instead of later
if not api_key:
    print("❌ ERROR: OPENAI_API_KEY not found in .env file!")
    print("   Please check your .env file is in the same folder as this notebook")

# STEP 3: Import LangChain components for AI/LLM work
from langchain_openai import ChatOpenAI  # Connect to OpenAI's chat models
from langchain_core.prompts import (
    PromptTemplate,              # Basic prompt template
    ChatPromptTemplate,          # Advanced prompt templates for chat
    MessagesPlaceholder          # Placeholder for conversation history
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import (
    BaseChatMessageHistory,      # Abstract base class for message history
    InMemoryChatMessageHistory   # Keep chat history in RAM (not persistent)
)

# STEP 4: Import YouTube transcript extraction library
import yt_dlp  # YouTube downloader - we use it to get captions
import re      # Regular expressions for text processing
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Split long texts smartly
from typing import Optional, Dict, List  # Type hints for better code clarity

# ============================================================================
# SUMMARY OF WHAT WE IMPORTED:
# ============================================================================
# CORE PROCESSING:  os, json, re, datetime, pandas
# WEB REQUESTS:     requests
# AI/LLM FRAMEWORK: langchain_openai, langchain_core
# YOUTUBE:          yt_dlp (extracts captions)
# ENVIRONMENT:      python-dotenv (reads secrets)
# TYPES:            Optional, Dict, List (tells Python what data we expect)
# ============================================================================

print("✓ All libraries imported successfully!")
print(f"✓ OpenAI API Key loaded: {'Yes' if api_key else 'No'}")


d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================================
# NOTE: This cell imports traceable from langsmith (already covered above)
# ============================================================================
# If you see an error here, it means LangSmith is not installed.
# That's OK! The system works without it. The try/except block in Cell 3
# handles this case already.
# ============================================================================

# Only run this if LangSmith is available
if LANGSMITH_AVAILABLE:
    try:
        from langsmith import traceable, Client as LangSmithClient
        from langsmith.run_trees import RunTree
        print("✓ LangSmith advanced features available")
    except ImportError:
        print("⚠️ LangSmith import failed (optional - continuing)")
else:
    print("⚠️ LangSmith not available (this is optional)")


In [ ]:
# ============================================================================
# CELL 3: CONFIGURE LANGSMITH (OPTIONAL - FOR ADVANCED MONITORING)
# ============================================================================
# What is LangSmith?
# It's a dashboard to monitor, trace, and debug your LLM pipelines.
# Think of it like: "Security camera for your AI system"
#
# This cell is OPTIONAL - everything works without it!
# But if you have a LangSmith account, it's incredibly useful.
# ============================================================================

# STEP 1: Try to import LangSmith (may not be installed)
# We use try/except because LangSmith is optional
LANGSMITH_AVAILABLE = False
try:
    from langsmith import traceable, Client as LangSmithClient  # Tracing decorators
    from langsmith.run_trees import RunTree                      # Advanced tracing
    LANGSMITH_AVAILABLE = True
    print("✓ LangSmith library found - tracing will be enabled")
except ImportError:
    print("⚠ LangSmith not installed - tracing will be disabled")
    print("  (This is fine! The system still works without it)")
    # Create dummy decorators so code doesn't break
    def traceable(*args, **kwargs):
        def decorator(fn):
            return fn
        return decorator

# STEP 2: Load LangSmith configuration from environment
# These should be in your .env file:
# LANGSMITH_API_KEY=lsv2_your_key_here
# LANGSMITH_PROJECT=YouTube-Transcript-Summarizer
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
langsmith_project = "YT-Transcript-Summarizer"

# STEP 3: Enable LangSmith tracing if API key exists
# Set environment variables that LangSmith looks for
if langsmith_api_key:
    os.environ['LANGSMITH_TRACEABLE_V2'] = "true"
    os.environ['LANGSMITH_PROJECT'] = langsmith_project
    print(f"✓ LangSmith tracing enabled for project: {langsmith_project}")
else:
    print("⚠ LangSmith API key not found - tracing disabled")
    print("  (To enable: Add LANGSMITH_API_KEY to your .env file)")

# STEP 4: Print summary of configuration
print("\n" + "="*60)
print("LANGSMITH CONFIGURATION SUMMARY")
print("="*60)
print(f"Library installed:      {LANGSMITH_AVAILABLE}")
print(f"API key configured:     {bool(langsmith_api_key)}")
print(f"Project name:           {langsmith_project}")
print(f"Tracing enabled:        {LANGSMITH_AVAILABLE and bool(langsmith_api_key)}")
print("="*60 + "\n")

# WHY THIS MATTERS:
# When tracing is enabled, EVERY LLM call gets logged to a dashboard.
# You can see:
# • Input prompts  (what we asked the AI)
# • Outputs        (what the AI responded)
# • Execution time (how long it took)
# • Token usage    (how much it cost)
# • Errors         (what went wrong)
# This is critical for production systems!


YT-Transcript-Summarizer is the project name


In [ ]:
# ============================================================================
# CELL 2b: VERIFY LANGSMITH CONFIGURATION (OPTIONAL)
# ============================================================================
# This cell checks if LangSmith is properly set up.
# LangSmith is optional but helpful for monitoring your AI system.
# ============================================================================

print("\n" + "="*60)
print("LANGSMITH CONFIGURATION CHECK")
print("="*60)

print(f"Is LangSmith library installed? {LANGSMITH_AVAILABLE}")
print(f"Is LANGSMITH_API_KEY set? {bool(langsmith_api_key)}")
print(f"Tracing enabled? {os.getenv('LANGSMITH_TRACEABLE_V2', 'false')}")
print(f"Project name: {os.getenv('LANGSMITH_PROJECT', 'Not set')}")

print("="*60)

if not langsmith_api_key:
    print("\n⚠️  LangSmith not fully configured")
    print("This is OPTIONAL - the system works without it!")
    print("\nTo enable LangSmith monitoring:")
    print("1. Visit https://smith.langchain.com")
    print("2. Create an account and get an API key")
    print("3. Add to your .env file: LANGSMITH_API_KEY=lsv2_...")
    print("4. Re-run this cell")
else:
    print("\n✅ LangSmith is configured!")
    print("All LLM calls will be traced and visible on the dashboard")


## 2️⃣ Transcript Extraction

Extract transcripts from YouTube videos using yt_dlp.

**Why do we need special parsing?**
Some YouTube captions are JSON structures like:
```json
{
  "events": [
    {
      "segs": [
        {"utf8": "Hello"},
        {"utf8": "world"}
      ]
    }
  ]
}
```

We need to extract the actual spoken text from these nested structures.

yt_dlp how does it work?
YouTube Downloader Library; We are using for metadata extraction - Captions/ Subtitles alone we are extracting;

In [ ]:
# ============================================================================
# CELL 4: EXTRACT YOUTUBE TRANSCRIPTS
# ============================================================================
# Goal: Download the actual text/transcript from YouTube videos
#
# Why is this hard?
# YouTube stores captions in special formats (JSON, VTT, etc.)
# We need to:
# 1. Download the caption file
# 2. Parse it (extract actual words from nested data structures)
# 3. Clean it up (remove timestamps, formatting)
# 4. Return clean text
#
# This is like: "Download a video's subtitles and extract just the words"
# ============================================================================

def _parse_json3_captions(json_text: str) -> str:
    """
    Parse YouTube's JSON3 caption format.
    
    YouTube sometimes stores captions like:
    {
        "events": [
            {
                "segs": [
                    {"utf8": "Hello"},
                    {"utf8": "world"}
                ]
            }
        ]
    }
    
    This function extracts just the text: "Hello world"
    
    Args:
        json_text (str): Raw JSON caption data from YouTube
        
    Returns:
        str: Plain text combining all captions
    """
    # Convert JSON text to a Python dictionary
    data = json.loads(json_text)
    
    # List to collect all text segments
    segs = []

    # Loop through each "event" (usually one event per speaking segment)
    for event in data.get("events", []):
        # Each event has "segs" (segments) which are individual words
        for seg in event.get("segs", []):
            # Extract the actual text from the "utf8" field
            text = seg.get("utf8")
            
            # Only add non-empty text (skip None or empty strings)
            if text:
                segs.append(text)
    
    # Join all words together with spaces: "Hello" + "world" = "Hello world"
    return " ".join(segs)


def extract_transcript(youtube_url: str) -> Optional[Dict[str, str]]:
    """
    Extract the transcript/captions from a YouTube video.
    
    WHAT THIS DOES:
    1. Opens the YouTube URL using yt_dlp
    2. Downloads the caption file (we DON'T download the video itself!)
    3. Extracts text from the caption file
    4. Cleans up formatting (removes timestamps, HTML tags)
    5. Returns the clean transcript
    
    EXAMPLE:
        result = extract_transcript("https://www.youtube.com/watch?v=HF2dVr7tHMI")
        print(result["title"])       # Video title
        print(result["transcript"])  # All captions as plain text
    
    Args:
        youtube_url (str): Full YouTube URL (must have captions!)
        
    Returns:
        Optional[Dict[str, str]]: 
            Returns: {"title": "...", "transcript": "..."}
            Or None if captions can't be found
    """
    
    # STEP 1: Configure yt_dlp (YouTube downloader)
    # These options tell yt_dlp what we want and what we don't want
    ydl_opts = {
        "quiet": True,                                    # Don't print verbose info
        "no_warnings": True,                              # Suppress warning messages
        "writesubtitles": True,                           # Extract manual captions
        "writeautomaticsub": True,                        # Extract auto-generated captions
        "skip_download": True,                            # Don't download video file (save bandwidth!)
        "subtitleslangs": ["en", "en-US", "en-GB", "en-IN"]  # Try these English variants
    }
    
    # STEP 2: Download video metadata and captions
    # yt_dlp returns a HUGE dictionary with all info about the video
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)

    # STEP 3: Extract title and captions list
    title = info.get("title", "Unknown Title")
    # Get captions - try manual first, then auto-generated, then empty dict
    subs = info.get("subtitles") or info.get("automatic_captions") or {}

    # STEP 4: Find an English caption track
    track = None  # Will store the caption file URL and metadata
    
    # Try to find captions in our preferred English variants
    for lang in ydl_opts["subtitleslangs"]:
        if lang in subs and subs[lang]:  # Does this language exist AND have captions?
            track = subs[lang][0]  # Get the first caption track for this language
            break

    # If our preferred language not found, just grab whatever language is available
    if not track and subs:
        first_lang = next(iter(subs))  # Get the first language key (e.g., "en", "fr", etc.)
        track = subs[first_lang][0]    # Get its first caption track

    # STEP 5: Download the caption file
    # This downloads the actual subtitle file (not the video!)
    raw_text = requests.get(track['url'], timeout=15).text

    # STEP 6: Parse the caption file based on its format
    # Different formats need different parsing:
    # - json3/srv3: Special JSON format (need to extract from nested structure)
    # - vtt: Video Text Track format (has timestamps we need to remove)
    ext = (track.get("ext") or "").lower()  # Get file type: "json3", "vtt", etc.

    if ext in {"json3", "srv3"}:
        # For JSON formats, use our special parser
        cleaned = _parse_json3_captions(raw_text)
    else:
        # For VTT format, we need to clean it manually
        cleaned = raw_text
        # Remove WEBVTT header (first few lines)
        cleaned = re.sub(r"WEBVTT.*?\n", "", cleaned, flags=re.DOTALL)
        # Remove timestamps like "00:01:23.456 --> 00:01:30.123"
        cleaned = re.sub(r"\d{2}:\d{2}:\d{2}\.\d{3}\s+-->\s+\d{2}:\d{2}:\d{2}\.\d{3}", "", cleaned)
        # Remove HTML tags like <v Name>
        cleaned = re.sub(r"<[^>]+>", "", cleaned)

    # STEP 7: Final cleanup - remove extra whitespace
    # Replace multiple spaces/newlines with single space
    cleaned = re.sub(r"\s+\n", "\n", cleaned)  # Remove spaces before newlines
    cleaned = re.sub(r"\s+", " ", cleaned)     # Replace multiple spaces with one
    cleaned = cleaned.strip()                   # Remove leading/trailing whitespace
    
    # STEP 8: Return the result
    return {"title": title, "transcript": cleaned}

# WHAT JUST HAPPENED:
# We built a function that:
# ✓ Downloads YouTube metadata (without downloading video!)
# ✓ Finds caption files in multiple language formats
# ✓ Extracts text from complex nested JSON structures
# ✓ Cleans up formatting to make it readable
# ✓ Returns a clean transcript ready for processing
#
# This is the foundation of the entire project!


In [10]:
test_url = "https://www.youtube.com/watch?v=HF2dVr7tHMI"
print("\n")
extract_transcript(test_url)



[youtube] Extracting URL: https://www.youtube.com/watch?v=HF2dVr7tHMI
[youtube] HF2dVr7tHMI: Downloading webpage
[youtube] HF2dVr7tHMI: Downloading android sdkless player API JSON
[youtube] HF2dVr7tHMI: Downloading web safari player API JSON
[youtube] HF2dVr7tHMI: Downloading m3u8 information
[info] HF2dVr7tHMI: Downloading subtitles: en


{'title': "Highlights from Microsoft Build 2025: Satya Nadella's Keynote Recap",
 'transcript': "We're taking really a systems approach, a platform approach, which you can expect from Microsoft across every layer of the stack. Whether it's GitHub and GitHub Copilot enabling an open ecosystem for the software development lifecycle. Microsoft 365 Copilot and Teams and Copilot Studio enabling agents for every role and business process. And an agent factory in Foundry, enabling you to build any AI app, any agent using any data, all running on world-class infrastructure. And all of this on a robust set of rails, for management, identity, and security. Ultimately, though, all of this is about creating opportunity to fuel your ambition."}

In [ ]:
# ============================================================================
# CELL: TEST TRANSCRIPT EXTRACTION
# ============================================================================
# This cell tests just the transcript extraction part.
# Use this to verify YouTube access and caption download works.
# ============================================================================

# Test URL - choose a YouTube video with English captions
test_url = "https://www.youtube.com/watch?v=HF2dVr7tHMI"

print("\n🔍 Testing transcript extraction...")
print(f"URL: {test_url}\n")

try:
    # Extract transcript
    result = extract_transcript(test_url)
    
    if result:
        print("✅ Transcript extraction successful!")
        print(f"\nTitle: {result['title']}")
        print(f"Transcript length: {len(result['transcript']):,} characters")
        print(f"\nFirst 300 characters of transcript:")
        print("-" * 60)
        print(result['transcript'][:300])
        print("-" * 60)
    else:
        print("❌ Failed to extract transcript")
        print("Possible reasons:")
        print("  • URL is invalid")
        print("  • Video has no captions")
        print("  • Network connection issue")
        
except Exception as e:
    print(f"❌ Error: {type(e).__name__}")
    print(f"Message: {str(e)}")
    print("\nDebug tips:")
    print("  • Check your internet connection")
    print("  • Verify the YouTube URL is valid")
    print("  • Ensure the video has English captions")


## 3️⃣ Text Preprocessing

Clean and prepare text for LLM processing.

In [ ]:
# ============================================================================
# CELL 5: TEXT PREPROCESSING (CLEANING)
# ============================================================================
# Before we send text to an LLM, we need to clean it.
# Why?
# - Extra spaces confuse the LLM
# - Special characters are meaningless
# - Consistent formatting helps AI understand better
#
# Think of it like: "Make sure your homework is neat before turning it in"
# ============================================================================

def preprocess_text(text: str) -> str:
    """
    Clean and normalize text for LLM processing.
    
    WHAT THIS DOES:
    1. Removes extra whitespace (multiple spaces → single space)
    2. Removes special/unusual characters (keeps punctuation)
    3. Fixes spacing after punctuation
    4. Returns clean, normalized text
    
    EXAMPLE:
        messy = "Hello    world!!!  How are    you?"
        clean = preprocess_text(messy)
        # Result: "Hello world! How are you?"
    
    Args:
        text (str): Raw, potentially messy text
        
    Returns:
        str: Cleaned, normalized text ready for LLM
    """
    
    # STEP 1: Replace any sequence of 2+ whitespace chars with single space
    # This includes: spaces, tabs, newlines, etc.
    # Before: "Hello     world"  (5 spaces)
    # After:  "Hello world"      (1 space)
    text = re.sub(r"\s+", " ", text)
    
    # STEP 2: Remove special/unusual characters but KEEP punctuation
    # This removes: emoji, accents, symbols, @#$%^& etc.
    # But keeps: letters, numbers, spaces, periods, commas, hyphens, !, ?
    # Pattern explanation:
    # [^\w\s.!?\-] means: "anything that is NOT (word char OR space OR punctuation)"
    # So we remove everything ELSE
    text = re.sub(r'[^\w\s.!?\-]', '', text)
    
    # STEP 3: Fix spacing after punctuation marks
    # Sometimes we get: "Hello.World" or "Hello. World" with irregular spacing
    # This ensures exactly one space after punctuation
    # Before: "Hello.  World" or "Hello.World"
    # After:  "Hello. World"
    text = re.sub(r'([.!?])\s+', r'\1 ', text)
    
    # STEP 4: Remove leading/trailing whitespace
    # Clean up any spaces at start/end of string
    return text.strip()

# EXAMPLE OF HOW THIS WORKS:
# ============================================================================
# Input:  "Hey!!!  Check  this    out???    Isn't   it   amazing?"
# After Step 1: "Hey!!!  Check  this    out???    Isn't   it   amazing?"
# After Step 2: "Hey Check this out Isnt it amazing"  (removed special chars)
# After Step 3: "Hey! Check this out? Isn't it amazing?"  (fixed punctuation spacing)
# Output: "Hey! Check this out? Isn't it amazing?"
# ============================================================================


In [ ]:
# ============================================================================
# CELL 6: TEXT CHUNKING (SPLITTING FOR LLM PROCESSING)
# ============================================================================
# Why do we need to chunk text?
#
# Problem: LLMs have token limits (usually 4,000 - 128,000 tokens)
#          A 1-hour video might have 100,000 tokens!
#          Sending it all at once would fail.
#
# Solution: Split text into chunks small enough for the LLM to handle
#           But with OVERLAP so the LLM understands context
#
# Think of it like: "Breaking a long book into chapters"
# Except we add overlap so each chapter mentions the end of the previous one.
# ============================================================================

def chunk_text(text: str, chunk_size: int = 2000, overlap: int = 200) -> List[str]:
    """
    Split text into overlapping chunks for LLM processing.
    
    WHY OVERLAP?
    Consider this passage:
        "...The cat was hungry. It ate the mouse. Then..."
    
    Without overlap:
        Chunk 1: "...The cat was hungry. It ate the..."
        Chunk 2: "...Then..."  ← Who is "it"? Context lost!
    
    With overlap:
        Chunk 1: "...The cat was hungry. It ate the mouse. Then..."
        Chunk 2: "...It ate the mouse. Then..."  ← Context preserved!
    
    PARAMETERS EXPLAINED:
    - chunk_size (2000): Each chunk has ~2000 characters
    - overlap (200): Chunks overlap by 200 characters
    
    So if we have:
        Chunk 1: chars 0-2000
        Chunk 2: chars 1800-3800  (started 200 chars before Chunk 1 ends)
        Chunk 3: chars 3600-5600
        ... and so on
    
    Args:
        text (str): Text to split
        chunk_size (int): Target size for each chunk (characters)
        overlap (int): How many characters should overlap between chunks
        
    Returns:
        List[str]: List of overlapping text chunks
    """
    
    # Create a RecursiveCharacterTextSplitter
    # This is a smart splitter from LangChain that:
    # 1. Tries to split at natural boundaries (\n\n = paragraph)
    # 2. Falls back to \n (newline) if needed
    # 3. Falls back to " " (space) if needed
    # 4. As last resort, splits mid-word
    # 
    # This avoids breaking in the middle of sentences!
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""]  # Try these, in order
    )
    
    # Actually split the text
    chunks = splitter.split_text(text)

    # Report success
    print(f"✓ Split text into {len(chunks)} chunks")
    print(f"  - Chunk size: ~{chunk_size} characters")
    print(f"  - Overlap: {overlap} characters")
    
    return chunks

# VISUAL EXAMPLE OF HOW CHUNKING WORKS:
# ============================================================================
# Original text (5000 chars):
# |===========CHUNK 1 (0-2000)===========|
#                         |===OVERLAP===|===========CHUNK 2 (1800-3800)===========|
#                                                               |===OVERLAP===|===========CHUNK 3 (3600-5000)===========|
#
# Benefits:
# ✓ Each chunk small enough for LLM to process
# ✓ Overlap preserves context between chunks
# ✓ No information is lost
# ✓ LLM can understand relationships across chunks
# ============================================================================


In [ ]:
# ============================================================================
# CELL: TEST TEXT PREPROCESSING
# ============================================================================
# This tests the text cleaning function in isolation.
# Useful for debugging and understanding how preprocessing works.
# ============================================================================

print("\n" + "="*70)
print("TESTING TEXT PREPROCESSING")
print("="*70)

# Create sample messy text to test with
sample_text = """
Hello!!!    This   is   a   test.    It  has   multiple    spaces   and   special chars!!!!
Check    this   out...   Amazing!!!   #awesome @testing $$$
Multiple  newlines

And weird   formatting.   Let's   see    how   it   cleans   up!
"""

print("\n📝 Original text:")
print("-" * 70)
print(repr(sample_text))  # repr() shows whitespace clearly
print("-" * 70)

# Run preprocessing
cleaned = preprocess_text(sample_text)

print("\n✅ Cleaned text:")
print("-" * 70)
print(repr(cleaned))  # repr() shows whitespace clearly
print("-" * 70)

print("\n📊 Comparison:")
print(f"Original length: {len(sample_text)} characters")
print(f"Cleaned length:  {len(cleaned)} characters")
print(f"Reduction:       {len(sample_text) - len(cleaned)} characters")

# Show what changed
print("\nWhat was removed:")
print("  ✓ Extra spaces (replaced multiple spaces with single space)")
print("  ✓ Special characters (#, @, $, %, etc.)")
print("  ✓ Leading/trailing whitespace")
print("  ✓ Multiple newlines consolidated")

print("\nWhat was kept:")
print("  ✓ Letters and numbers")
print("  ✓ Punctuation (. ! ?)")
print("  ✓ Hyphens (-)")
print("  ✓ Spaces between words")


## 4️⃣ LangChain Prompts

Create prompt templates for different summarization styles.

In [ ]:
# ============================================================================
# CELL 7: CREATE PROMPT TEMPLATES
# ============================================================================
# What is a prompt template?
#
# It's a reusable template for asking questions to the LLM.
# Instead of writing the full prompt each time, we create templates
# and fill in the {placeholders} with actual data.
#
# Example:
#   Template: "Summarize this text in 3 sentences:\n{text}"
#   Data: "The cat was black and fluffy..."
#   Result: "Summarize this text in 3 sentences:\nThe cat was black and fluffy..."
#
# Why templates?
# ✓ Consistent formatting (same prompt structure each time)
# ✓ Easy to update (change template once, affects all uses)
# ✓ Reusable (use same template for different texts)
# ✓ Professional (proven to improve LLM quality)
# ============================================================================

# PROMPT 1: CONCISE SUMMARY
# Goal: Get a short summary (3-4 sentences)
# Use case: Quick overview, Twitter-style summary
concise_summary_prompt = PromptTemplate(
    input_variables=["text"],  # This prompt takes "text" as input
    template="""You are an expert content summarizer. Provide a concise summary 
of the following text in 3-4 sentences. Focus on the main points and key takeaways.

Text:
{text}

Summary:"""
)

# PROMPT 2: DETAILED SUMMARY
# Goal: Get a detailed analysis with structure
# Use case: Full understanding, learning, documentation
detailed_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Analyze the following text and provide:
1. A brief overview (2-3 sentences)
2. Key points (4-6 bullet points)
3. Main conclusions

Text:
{text}

Analysis:"""
)

# PROMPT 3: STRUCTURED SUMMARY
# Goal: Break content into What/Why/How/Outcomes
# Use case: Problem solving, learning frameworks, teaching
structured_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Provide a structured summary of the following text with:
- **What**: What is the main topic?
- **Why**: Why is it important?
- **How**: Key methods or approaches mentioned
- **Outcomes**: What are the results or conclusions?

Text:
{text}

Structured Summary:"""
)

# ============================================================================
# PROMPT ENGINEERING TIPS:
# ============================================================================
# Good prompts have these characteristics:
#
# 1. ROLE DEFINITION
#    ✓ "You are an expert content summarizer"
#    ✗ "Summarize this"
#    Why? Gives LLM context for how to respond
#
# 2. CLEAR INSTRUCTIONS
#    ✓ "Provide a concise summary in 3-4 sentences"
#    ✗ "Make a summary"
#    Why? Specifies FORMAT and LENGTH
#
# 3. OUTPUT FORMAT
#    ✓ "Provide: 1) Overview 2) Key points 3) Conclusions"
#    ✗ "Tell me about it"
#    Why? Structures the response for easier parsing
#
# 4. EXAMPLES (sometimes)
#    ✓ "Format: Topic: [topic] Importance: [importance]"
#    ✗ "Write something useful"
#    Why? Shows exactly what format you want
# ============================================================================

print("✓ Created 3 prompt templates:")
print("  1. Concise summary (3-4 sentences)")
print("  2. Detailed summary (overview + key points + conclusions)")
print("  3. Structured summary (What/Why/How/Outcomes)")


## 5️⃣ YouTubeSummarizer Class

Build the main summarizer class that uses LangChain.

In [ ]:
# ============================================================================
# CELL 8: YOUTUBE SUMMARIZER CLASS
# ============================================================================
# Why use a class?
#
# A class is like a blueprint for an object that does one thing well.
# Our class: YouTubeSummarizer
# Its job: Extract transcripts and summarize them
#
# Benefits:
# ✓ Reusable: Create one summarizer, use it for many videos
# ✓ Organized: All related code in one place
# ✓ Maintainable: Easy to update and debug
# ✓ Professional: Standard way to organize code
#
# Think of it like: A tool factory. We create the factory once,
# then use it to create many summaries.
# ============================================================================

class YouTubeSummarizer:
    """
    Main class for extracting and summarizing YouTube transcripts.
    
    WHAT THIS CLASS DOES:
    1. Connects to OpenAI's LLM via LangChain
    2. Summarizes individual text chunks
    3. Combines chunk summaries into a final summary
    4. Handles the entire pipeline for one video
    
    USAGE EXAMPLE:
        # Create one summarizer instance
        summarizer = YouTubeSummarizer(api_key="sk-...")
        
        # Use it to summarize any transcript
        result = summarizer.summarize_full_transcript(
            transcript="Long text here...",
            style="structured"
        )
        
        print(result["overall_summary"])
    """
    
    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-3.5-turbo"):
        """
        Initialize the summarizer with an LLM.
        
        Args:
            api_key (Optional[str]): OpenAI API key. If None, uses env var.
            model (str): Which OpenAI model to use. Default: "gpt-3.5-turbo"
        """
        # Create a ChatOpenAI instance (LangChain's interface to OpenAI)
        # Parameters:
        # - api_key: Your OpenAI authentication key
        # - model: Which model to use (gpt-3.5-turbo, gpt-4, etc.)
        # - temperature: 0-1 scale for creativity (0.5 = balanced)
        self.llm = ChatOpenAI(
            api_key=api_key,
            model=model,
            temperature=0.5  # Balanced: not too creative, not too robotic
        )
        
        print(f"✓ Summarizer initialized with {model}")

    def summarize_chunk(self, text: str, style: str = "concise") -> str:
        """
        Summarize a single text chunk using the appropriate prompt.
        
        WHAT THIS DOES:
        1. Picks the right prompt template based on style
        2. Sends text + prompt to the LLM
        3. Gets back a summary
        4. Cleans it up and returns it
        
        Args:
            text (str): Text chunk to summarize (up to ~2000 chars)
            style (str): Summary style - "concise", "detailed", or "structured"
            
        Returns:
            str: The LLM's summary of the text
        """
        
        # STEP 1: Create a dictionary mapping styles to prompts
        # This lets us pick the right prompt based on user preference
        prompts = {
            "concise": concise_summary_prompt,      # 3-4 sentences
            "detailed": detailed_summary_prompt,    # Overview + bullets + conclusions
            "structured": structured_summary_prompt # What/Why/How/Outcomes
        }

        # STEP 2: Get the appropriate prompt
        # Use "concise" if style not found (default)
        prompt = prompts.get(style, concise_summary_prompt)
        
        # STEP 3: Create a chain using LCEL (LangChain Expression Language)
        # Chain = Prompt → LLM → Output
        # The | operator means "pipe to" or "send output to"
        # So this means: "Feed text to prompt, then send result to LLM"
        chain = prompt | self.llm

        # STEP 4: Send the text through the chain
        # invoke() executes the chain and returns the result
        result = chain.invoke({"text": text})

        # STEP 5: Extract and clean the response
        # The result is an LLM message object, we need result.content (the text)
        # .strip() removes leading/trailing whitespace
        return result.content.strip()
    
    def summarize_full_transcript(
            self,
            transcript: str,
            style: str = "structured",
            chunk_size: int = 2000
    ) -> Optional[Dict]:
        """
        Summarize an entire transcript end-to-end.
        
        THE FULL PIPELINE:
        1. Clean the transcript (remove extra whitespace, special chars)
        2. Split into chunks (overlapping pieces small enough for LLM)
        3. Summarize each chunk (send each to LLM)
        4. Combine summaries (merge chunk summaries)
        5. Summarize again (create final overall summary)
        6. Return results (packaged nicely)
        
        EXAMPLE:
            result = summarizer.summarize_full_transcript(
                transcript="Long video transcript...",
                style="structured",
                chunk_size=2000
            )
            
            print(result["overall_summary"])  # Final summary
            for i, chunk_sum in enumerate(result["chunk_summaries"]):
                print(f"Chunk {i}: {chunk_sum}")
        
        Args:
            transcript (str): Full video transcript (can be very long)
            style (str): Summary style (default: "structured")
            chunk_size (int): Characters per chunk (default: 2000)
            
        Returns:
            Optional[Dict]: Results dict, or None if something failed
        """
        
        # STEP 1: Preprocess/clean the transcript
        # Remove extra whitespace, special chars, fix formatting
        cleaned_text = preprocess_text(transcript)

        # STEP 2: Split into chunks
        # This creates overlapping pieces, each ~2000 chars
        chunks = chunk_text(cleaned_text, chunk_size=chunk_size)

        # STEP 3: Check if we got any chunks (shouldn't happen, but just in case)
        if not chunks:
            return None

        # STEP 4: Summarize each chunk
        # This is where we send each piece to OpenAI
        chunk_summaries = []
        
        for i, chunk in enumerate(chunks):
            # Send this chunk to the LLM
            summary = self.summarize_chunk(chunk, style=style)
            
            # Only add non-empty summaries
            if summary:
                chunk_summaries.append(summary)
                # Optional: Print progress
                print(f"  ✓ Summarized chunk {i+1}/{len(chunks)}")

        # STEP 5: Combine all chunk summaries
        # Join them with double newlines for readability
        combined_text = "\n\n".join(chunk_summaries)
        
        # STEP 6: Create a final overall summary
        # Summarize the summaries! This gives us the big picture
        overall_summary = self.summarize_chunk(combined_text, style="concise")

        # STEP 7: Package and return results
        return {
            "chunk_summaries": chunk_summaries,
            "overall_summary": overall_summary,
            "num_chunks": len(chunks),
            "style": style
        }

print("✓ YouTubeSummarizer class created and ready to use!")


## 6️⃣ Complete Pipeline

Integrate everything into one end-to-end pipeline.

In [26]:
# TODO: Create main pipeline function
#
# STEP 1: Import traceable decorator (if LangSmith available)
# HINT: If LANGSMITH_AVAILABLE:
#           from langsmith import traceable
#       else:
#           def traceable(*args, **kwargs):
#               def decorator(fn):
#                   return fn
#               return decorator
#
# STEP 2: Create summarize_youtube_video function
# HINT: @traceable(name="youtube_summarizer_pipeline")
#       def summarize_youtube_video(youtube_url: str, style: str = "structured", api_key: Optional[str] = None):
#   - Extract transcript: transcript_data = extract_transcript(youtube_url)
#   - If None, return None
#   - Create summarizer: summarizer = YouTubeSummarizer(api_key=api_key)
#   - Summarize: results = summarizer.summarize_full_transcript(transcript, style=style)
#   - Return dict with title, video_url, timestamp, summaries
#
# STEP 3: Create display_results function
# HINT: @traceable(name="display_summarization_results")
#       def display_results(results: Dict):
#   - Print title, URL, timestamp
#   - Print overall summary
#   - Print each chunk summary
#
# STEP 4: Print confirmation
#
# Write your code below:
# ==========================================


# =============================================================================
# 🚀 END-TO-END PIPELINE FUNCTIONS
# =============================================================================
# Purpose: Integrate ALL components into one function that does everything
# Input: YouTube URL
# Output: Complete transcript + multiple summaries
#
# This is what students will call to use your system!

from langsmith import traceable

@traceable(
    name="youtube_summarizer_pipeline",
    description="Extract transcript and generate LLM summary (LangSmith traced)"
)
def summarize_youtube_video(
    youtube_url: str,
    style: str = "structured",
    api_key: Optional[str] = None
) -> Optional[Dict]:
    """
    Extract YouTube transcript and summarize it in one shot.
    
    WHAT THIS DOES (The Full Pipeline):
    1. Extract → Download transcript from YouTube using yt_dlp
    2. Preprocess → Clean the text (remove formatting, extra spaces)
    3. Chunk → Split into manageable pieces
    4. Summarize → Send each chunk to OpenAI's LLM
    5. Combine → Merge all summaries into one polished output
    6. Return → Organized dictionary with all results
    
    VISUAL FLOW:
    YouTube URL
        ↓
    extract_transcript() → {title, transcript}
        ↓
    Preprocess → Clean text
        ↓
    Chunk → Split into 10 pieces
        ↓
    summarize_chunk() × 10 → Summarize each (parallel-friendly!)
        ↓
    Combine → Join 10 summaries
        ↓
    Final summary → Summarize the summaries
        ↓
    Return → Complete results
    
    PRODUCTION READY FEATURES:
    • Error handling (returns None if transcript not found)
    • Progress tracking (prints status messages)
    • LangSmith tracing (if enabled, sends data to monitoring dashboard)
    • Type hints (helps developers understand what to pass in)
    
    Args:
        youtube_url (str): Full YouTube URL, e.g., "https://www.youtube.com/watch?v=..."
        style (str): Summary style - 'concise', 'detailed', or 'structured' (default: 'structured')
        api_key (Optional[str]): OpenAI API key (uses env var if not provided)
        
    Returns:
        Optional[Dict]: Dictionary with structure:
        {
            "title": "Video title",
            "video_url": "https://...",
            "timestamp": "2025-01-04T10:30:45.123Z",
            "transcript_summary": {
                "original_length": 50000,  # characters
                "num_chunks": 10,
            },
            "summaries": {
                "overall_summary": "...",
                "chunk_summaries": ["...", "...", "..."],
                "style": "structured"
            }
        }
        Or None if extraction/summarization fails
    
    THE @traceable DECORATOR:
    This is LangSmith magic. When you decorate a function with @traceable:
    • Every call gets logged to LangSmith dashboard
    • Input/output recorded automatically
    • Execution time measured
    • Cost calculated
    • Errors captured
    
    This is observability—seeing what your AI system is actually doing!
    """
    
    # =========================================================================
    # STEP 1: Extract the transcript
    # =========================================================================
    transcript_data = extract_transcript(youtube_url)
    if not transcript_data:
        # If extraction failed (no captions, URL invalid, etc.), return None
        return None
    
    # Extract title and transcript from the result dictionary
    title = transcript_data["title"]
    transcript = transcript_data["transcript"]
    
    # =========================================================================
    # STEP 2: Create summarizer and process
    # =========================================================================
    # Initialize the summarizer with OpenAI (reuses one LLM instance)
    summarizer = YouTubeSummarizer(api_key=api_key)
    
    # Run the full summarization pipeline
    results = summarizer.summarize_full_transcript(
        transcript, 
        style=style, 
        chunk_size=2000
    )
    
    # If summarization failed, return None
    if not results:
        return None
    
    # =========================================================================
    # STEP 3: Organize and return results
    # =========================================================================
    # Structure the output as a nice dictionary with metadata
    return {
        "title": title,
        "video_url": youtube_url,
        "timestamp": datetime.now().isoformat(),  # When did we run this?
        "transcript_summary": {
            "original_length": len(transcript),
            "num_chunks": results["num_chunks"],
        },
        "summaries": {
            "overall_summary": results["overall_summary"],
            "chunk_summaries": results["chunk_summaries"],
            "style": results["style"],
        },
    }


@traceable(
    name="display_summarization_results",
    description="Pretty-print summarization output"
)
def display_results(results: Dict) -> None:
    """
    Pretty-print summarization results in a readable format.
    
    WHY THIS FUNCTION?
    Don't just print the raw dictionary. Format it nicely for humans!
    This is good practice in any data processing pipeline:
    • Extract
    • Process
    • Present
    
    The "present" step is often overlooked but important for:
    ✅ Stakeholders (they don't read raw JSON)
    ✅ Debugging (formatted output easier to scan)
    ✅ Reports (looks professional)
    
    Args:
        results (Dict): Output from summarize_youtube_video()
    """
    print("\n=== SUMMARIZATION RESULTS ===")
    print(f"Title: {results['title']}")
    print(f"URL:   {results['video_url']}")
    print(f"When:  {results['timestamp']}")
    print(f"Chunks: {len(results['summaries']['chunk_summaries'])} (style: {results['summaries']['style']})\n")
    
    # Print the overall summary (one-liner for quick reading)
    print("Overall summary:\n", results['summaries']['overall_summary'])
    
    # Print individual chunk summaries (for detailed reading)
    print("\nChunk summaries:")
    for i, summary in enumerate(results['summaries']['chunk_summaries'], 1):
        print(f"\n[Chunk {i}]\n{summary}")

    
print("✓ Pipeline functions (traced) ready for teaching")





✓ Pipeline functions (traced) ready for teaching


## 7️⃣ Usage Examples

Test the complete pipeline.

In [ ]:
# ============================================================================
# CELL 9: TEST PREPROCESSING ON REAL DATA
# ============================================================================
# What are we testing?
# 1. Extract a transcript from a YouTube video
# 2. Preprocess it (clean it up)
# 3. Chunk it (split into pieces)
# 4. Display before/after stats
#
# Why test?
# Before using something in production, make sure it works!
# This is like a "dry run" or "practice round"
# ============================================================================

# The URL we'll test with
test_url = "https://www.youtube.com/watch?v=HF2dVr7tHMI"

print("="*70)
print("TESTING TRANSCRIPT EXTRACTION, PREPROCESSING, AND CHUNKING")
print("="*70)

if test_url:
    # STEP 1: Extract the transcript
    print("\n1️⃣ EXTRACTING TRANSCRIPT...")
    result = extract_transcript(test_url)

    if result:
        # Get title and transcript
        title = result['title']
        transcript = result['transcript']
        
        print(f"✓ Title: {title}")
        print(f"✓ Transcript length: {len(transcript):,} characters")
        
        # STEP 2: Show a preview of the raw transcript
        print(f"\nPreview of raw transcript (first 200 chars):")
        print(f"  {transcript[:200]}...")
        
        # STEP 3: Preprocess the transcript
        print("\n2️⃣ PREPROCESSING TEXT...")
        cleaned = preprocess_text(transcript)
        print(f"✓ Cleaned transcript length: {len(cleaned):,} characters")
        print(f"✓ Reduction: {len(transcript) - len(cleaned):,} characters removed")
        print(f"\nPreview of cleaned text (first 200 chars):")
        print(f"  {cleaned[:200]}...")
        
        # STEP 4: Chunk the text
        print("\n3️⃣ CHUNKING TEXT...")
        chunks = chunk_text(cleaned, chunk_size=500)
        
        # STEP 5: Display statistics about chunks
        print("\n4️⃣ CHUNK STATISTICS...")
        chunk_sizes = [len(c) for c in chunks]
        print(f"Total chunks: {len(chunks)}")
        print(f"Average chunk size: {sum(chunk_sizes) // len(chunks):,} characters")
        print(f"Smallest chunk: {min(chunk_sizes):,} characters")
        print(f"Largest chunk: {max(chunk_sizes):,} characters")
        
        print(f"\nPreview of first chunk (500 chars max):")
        print(f"  {chunks[0][:500]}...")
        
        if len(chunks) > 1:
            print(f"\nPreview of second chunk (to show overlap):")
            print(f"  {chunks[1][:200]}...")
    else:
        print("❌ Could not extract transcript.")
        print("   Reasons: Invalid URL, no captions, network error")
        print("   Try a different video or check your internet connection.")
else:
    print("❌ test_url not set")

print("\n" + "="*70)


[youtube] Extracting URL: https://www.youtube.com/watch?v=HF2dVr7tHMI
[youtube] HF2dVr7tHMI: Downloading webpage
[youtube] HF2dVr7tHMI: Downloading android sdkless player API JSON
[youtube] HF2dVr7tHMI: Downloading web safari player API JSON
[youtube] HF2dVr7tHMI: Downloading m3u8 information
[info] HF2dVr7tHMI: Downloading subtitles: en
Highlights from Microsoft Build 2025: Satya Nadella's Keynote Recap
642
630
Were taking really a systems approach a platform approach which you can expect from Microsoft across every layer of the stack. Whether its GitHub and GitHub Copilot enabling an open ecosystem for the 
2
First chunk (500 chars): Were taking really a systems approach a platform approach which you can expect from Microsoft across every layer of the stack. Whether its GitHub and ...

Chunk size analysis:
  Average chunk size: 315 characters
  Smallest chunk: 327 chars
  Largest chunk: 500 chars


In [ ]:
# =============================================================================
# 🚀 FULL PIPELINE EXECUTION
# =============================================================================
# Configuration: Set these variables to control the pipeline

# The YouTube video to summarize
YOUTUBE_URL = "https://www.youtube.com/watch?v=HF2dVr7tHMI"

# How to summarize: "concise" (3-4 sentences), "detailed" (with bullets), or "structured" (What/Why/How/Outcomes)
SUMMARY_STYLE = "structured"  # Options: "concise", "detailed", "structured"

print(f"\n📺 Summarizing: {YOUTUBE_URL}")
print(f"📋 Style: {SUMMARY_STYLE}")
print("=" * 60)

# =============================================================================
# RUN THE FULL PIPELINE
# =============================================================================
# This single function call does everything:
# extract → preprocess → chunk → summarize → combine
results = summarize_youtube_video(
    youtube_url=YOUTUBE_URL,
    style=SUMMARY_STYLE,
    api_key=None  # Uses OPENAI_API_KEY env variable
)

# Display the results if successful
if results:
    display_results(results)
    
    # OPTIONAL: Save results to file
    # Uncomment below to save as JSON for later use
    # import json
    # with open("summary_results.json", "w") as f:
    #     json.dump(results, f, indent=2)
    # print("\n✓ Results saved to summary_results.json")
else:
    print("❌ Failed to summarize. Check URL and API key.")

In [ ]:
# ============================================================================
# CELL: TEST INDIVIDUAL COMPONENTS (BONUS TESTING)
# ============================================================================
# This cell is for experimenting with individual functions.
# Use it to test and debug each part of the system separately.
#
# Examples of what you can test:
# ============================================================================

print("="*70)
print("BONUS: TESTING INDIVIDUAL COMPONENTS")
print("="*70)

# EXAMPLE 1: Test the summarizer with a simple text
print("\n1️⃣ Testing summarizer with sample text...")

sample_transcript = """
Artificial Intelligence is transforming industries. AI systems can now recognize 
images, understand language, and make predictions. Companies are investing heavily 
in AI. ChatGPT became popular in 2022. It can write essays, answer questions, and 
help with coding. AI has limitations too. It can be biased, it uses lots of energy, 
and it can produce false information. As AI becomes more powerful, we need to think 
about ethics and safety. The future of AI is exciting but requires careful consideration.
"""

print(f"Sample text length: {len(sample_transcript)} characters\n")

try:
    # Create a summarizer
    summarizer = YouTubeSummarizer()
    
    # Test the chunk summarization
    print("Summarizing sample text...\n")
    summary = summarizer.summarize_chunk(sample_transcript, style="concise")
    
    print("✅ Summary result:")
    print("-" * 70)
    print(summary)
    print("-" * 70)
    
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    print("\nTroubleshooting:")
    print("  • Check OPENAI_API_KEY in .env")
    print("  • Verify API key is valid and has credits")
    print("  • Check internet connection")

# EXAMPLE 2: Test preprocessing then chunking
print("\n2️⃣ Testing preprocess + chunk pipeline...\n")

test_long_text = sample_transcript * 3  # Make it longer

cleaned = preprocess_text(test_long_text)
chunks = chunk_text(cleaned, chunk_size=300)

print(f"Original: {len(test_long_text)} chars")
print(f"Cleaned: {len(cleaned)} chars")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunk size: {len(cleaned) // len(chunks)} chars")

# EXAMPLE 3: Try different prompt styles
print("\n3️⃣ Testing different summary styles...\n")

try:
    summarizer = YouTubeSummarizer()
    
    styles = ["concise", "detailed", "structured"]
    
    for style in styles:
        print(f"Style: {style.upper()}")
        summary = summarizer.summarize_chunk(sample_transcript, style=style)
        print(f"  {summary[:100]}...\n")
        
except Exception as e:
    print(f"⚠️ Error: {e}")

print("="*70)
print("\nFEEL FREE TO ADD MORE TESTS HERE!")
print("Try different URLs, texts, and configurations.")
print("="*70)


## 📚 Key Concepts Learned

### LangChain Components:
- **PromptTemplate**: Structured templates for LLM inputs
- **ChatOpenAI**: Integration with OpenAI's models
- **LCEL (LangChain Expression Language)**: Chain prompts with LLMs using `|`

### Text Processing:
- **Text Cleaning**: Remove noise and formatting
- **Chunking**: Split large texts into manageable pieces with overlap
- **RecursiveCharacterTextSplitter**: Smart text splitting at natural boundaries

### YouTube Extraction:
- **yt_dlp**: Extract video metadata and captions
- **Subtitle Formats**: Handle JSON and VTT formats
- **Error Handling**: Gracefully handle missing captions

### LangSmith (Optional):
- **Tracing**: Monitor all LLM calls automatically
- **Observability**: Track tokens, costs, latency
- **Debugging**: Inspect inputs/outputs of each step

## 🚀 Next Steps & Challenges

### Challenges to Try:
1. ✅ Add multi-language support (translate summaries)
2. ✅ Export summaries as PDF or Markdown files
3. ✅ Build a Streamlit UI for easy interaction
4. ✅ Add caching to avoid re-processing same URLs
5. ✅ Create custom prompts for specific domains (tech, education, news)
6. ✅ Implement batch processing for playlist URLs
7. ✅ Add quality metrics (summary length, coherence checks)

### Advanced Topics:
- Try different LLM providers (Anthropic Claude, Google Gemini)
- Implement streaming responses for real-time feedback
- Add vector embeddings for semantic search across summaries
- Build a knowledge base from multiple video summaries